# HRRR Precipitation Forecast Download

This notebook demonstrates how to download and process HRRR (High-Resolution Rapid Refresh) precipitation forecast data using the `PrecipHrrr` class in ras-commander.

**HRRR** is NOAA's operational numerical weather prediction model with:
- **3km horizontal resolution** over CONUS
- **Hourly forecast cycles** (00z through 23z)
- **18-hour forecast horizon** for standard cycles
- **48-hour forecast horizon** for extended cycles (00z, 06z, 12z, 18z)
- **Data source**: NOAA NOMADS server (publicly available, no account required)

HRRR is particularly valuable for:
- Short-range flood forecasting
- Real-time operational HEC-RAS model forcing
- Ensemble precipitation inputs for hydraulic models

## Setup and Imports

In [1]:
# =============================================================================
# DEVELOPMENT MODE TOGGLE
# =============================================================================
# Set USE_LOCAL_SOURCE based on your setup:
#   True  = Use local source code (for developers editing ras-commander)
#   False = Use pip-installed package (for users)
# =============================================================================

USE_LOCAL_SOURCE = True  # <-- TOGGLE THIS

# -----------------------------------------------------------------------------
if USE_LOCAL_SOURCE:
    import sys
    from pathlib import Path
    local_path = str(Path.cwd().parent)  # Parent of examples/ = repo root
    if local_path not in sys.path:
        sys.path.insert(0, local_path)  # Insert at position 0 = highest priority
    print(f"LOCAL SOURCE MODE: Loading from {local_path}/ras_commander")
else:
    print("PIP PACKAGE MODE: Loading installed ras-commander")

# Verify which version loaded
import ras_commander
print(f"Loaded: {ras_commander.__file__}")

LOCAL SOURCE MODE: Loading from G:\GH\ras-commander-wt-qpkit/ras_commander


G:\GH\qpkit\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded: G:\GH\ras-commander-wt-qpkit\ras_commander\__init__.py


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from ras_commander.precip import PrecipHrrr

print("Imports complete.")

Imports complete.


## What You'll Learn

- How to query HRRR data availability on NOAA NOMADS
- How to download the latest HRRR forecast cycle automatically
- How to download a specific date and cycle
- How to extract precipitation data from GRIB2 files to xarray
- How to compute basin-average precipitation time series
- HRRR cycle timing and forecast horizon reference

## HRRR Overview

The High-Resolution Rapid Refresh (HRRR) model runs every hour at 3km resolution. Understanding cycle timing and data latency is important for operational use:

| Cycle | Type | Forecast Horizon | Typical Availability |
|-------|------|-----------------|---------------------|
| 00z, 06z, 12z, 18z | Extended | 48 hours | ~2 hours after cycle |
| All other hours | Standard | 18 hours | ~2 hours after cycle |

**File format**: GRIB2 (`.grib2`)

**Key precipitation variable**: `APCP` (Accumulated Precipitation, kg/m²)

**Data server**: [NOAA NOMADS](https://nomads.ncep.noaa.gov/pub/data/nccf/com/hrrr/prod/)

**Retention**: NOMADS retains approximately 48 hours of recent HRRR output. For archived HRRR data, use the NOAA HRRR archive on AWS.

In [3]:
# Get HRRR system information
info = PrecipHrrr.get_info()
print("HRRR System Information:")
print(f"  Data source: {info.get('base_url', 'NOAA NOMADS')}")
print(f"  Resolution: {info.get('spatial_resolution', '3km')}")
print(f"  Standard forecast: {info.get('forecast_horizon_standard', '18 hours')}")
print(f"  Extended forecast: {info.get('forecast_horizon_extended', '48 hours')}")
print(f"  Update frequency: {info.get('update_frequency', 'Hourly')}")
print(f"  CONUS bounds: {info.get('bounds', 'Full CONUS')}")

HRRR System Information:
  Data source: https://nomads.ncep.noaa.gov/pub/data/nccf/com/hrrr/prod
  Resolution: 3 km
  Standard forecast: 18 hours
  Extended forecast: 48 hours (00z, 06z, 12z, 18z cycles)
  Update frequency: Hourly (every cycle)
  CONUS bounds: (-134.1, 21.1, -60.9, 52.6)


## Example 1: Single Forecast Cycle

In this example we use `get_latest_forecast()` to automatically find and download the most recent available HRRR cycle. This is the recommended approach for operational workflows.

### Check Availability

Before downloading, verify that the desired HRRR cycle is available on NOMADS. This is useful for operational monitoring and retry logic.

In [4]:
# Check if the latest HRRR cycle is available
try:
    available = PrecipHrrr.check_availability()
    print(f"Latest HRRR cycle available: {available}")

    # Check a specific date/cycle
    yesterday = datetime.now() - timedelta(days=1)
    available_12z = PrecipHrrr.check_availability(
        date=yesterday.strftime("%Y-%m-%d"),
        cycle=12
    )
    print(f"Yesterday's 12z cycle available: {available_12z}")
    print(f"  (Checked date: {yesterday.strftime('%Y-%m-%d')})")
except Exception as e:
    print(f"Availability check requires internet: {e}")
    print("\nExpected output:")
    print("  Latest HRRR cycle available: True")
    print("  Yesterday's 12z cycle available: True")

Latest HRRR cycle available: True


Yesterday's 12z cycle available: False
  (Checked date: 2026-06-23)


### Download Latest HRRR Forecast

`get_latest_forecast()` searches backwards through recent cycles (controlled by `max_lookback_hours`) to find the most recently published HRRR data. This handles the ~2 hour latency between cycle time and data availability.

In [5]:
# Download latest HRRR cycle (18 hours ahead)
output_dir = Path("example_data/hrrr_single")
output_dir.mkdir(parents=True, exist_ok=True)

try:
    grib_files = PrecipHrrr.get_latest_forecast(
        output_dir=output_dir,
        hours=18,
        max_lookback_hours=6  # Look back up to 6 hours for available data
    )
    print(f"Downloaded {len(grib_files)} GRIB2 files")
    print(f"\nFirst 5 files:")
    for f in grib_files[:5]:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f"  {f.name} ({size_mb:.0f} MB)")
except Exception as e:
    print(f"Download requires internet access to NOAA NOMADS: {e}")
    print("\nExpected output:")
    print("  Downloaded 18 GRIB2 files")
    print("  hrrr.t12z.wrfsubhf01.grib2 (~100-150 MB)")
    print("  hrrr.t12z.wrfsubhf02.grib2 (~100-150 MB)")
    print("  ...")
    grib_files = []

2026-06-24 22:03:31 - ras_commander.precip.PrecipHrrr - INFO - Found available cycle: 2026-06-25 00z


2026-06-24 22:03:31 - ras_commander.precip.PrecipHrrr - INFO - Downloading: hrrr.t00z.wrfsubhf01.grib2 (hour 1/18)


2026-06-24 22:03:44 - ras_commander.precip.PrecipHrrr - INFO - Downloading: hrrr.t00z.wrfsubhf02.grib2 (hour 2/18)


2026-06-24 22:03:53 - ras_commander.precip.PrecipHrrr - INFO - Downloading: hrrr.t00z.wrfsubhf03.grib2 (hour 3/18)


2026-06-24 22:04:01 - ras_commander.precip.PrecipHrrr - INFO - Downloading: hrrr.t00z.wrfsubhf04.grib2 (hour 4/18)


...

2026-06-24 22:05:52 - ras_commander.precip.PrecipHrrr - INFO - Downloading: hrrr.t00z.wrfsubhf16.grib2 (hour 16/18)


2026-06-24 22:06:01 - ras_commander.precip.PrecipHrrr - INFO - Downloading: hrrr.t00z.wrfsubhf17.grib2 (hour 17/18)


2026-06-24 22:06:12 - ras_commander.precip.PrecipHrrr - INFO - Downloading: hrrr.t00z.wrfsubhf18.grib2 (hour 18/18)


2026-06-24 22:06:21 - ras_commander.precip.PrecipHrrr - INFO - Downloaded 18 HRRR files for 20260625 00z (hours 1-18)


Downloaded 18 GRIB2 files

First 5 files:
  hrrr.t00z.wrfsubhf01.grib2 (198 MB)
  hrrr.t00z.wrfsubhf02.grib2 (186 MB)
  hrrr.t00z.wrfsubhf03.grib2 (174 MB)
  hrrr.t00z.wrfsubhf04.grib2 (167 MB)
  hrrr.t00z.wrfsubhf05.grib2 (166 MB)


### Extract Precipitation Data

`extract_precipitation()` reads the GRIB2 files and returns an `xarray.Dataset` containing the precipitation variable(s). An optional `bounds` parameter (west, south, east, north) clips the data to a region of interest, which significantly reduces memory usage.

In [6]:
if grib_files:
    try:
        # Extract precipitation for a specific region (Bald Eagle Creek, PA)
        bounds = (-77.9, 40.8, -77.3, 41.1)  # west, south, east, north

        precip_ds = PrecipHrrr.extract_precipitation(
            grib_files=grib_files,
            bounds=bounds
        )
        print("Extracted precipitation dataset:")
        print(f"  Variables: {list(precip_ds.data_vars)}")
        print(f"  Dimensions: {dict(precip_ds.dims)}")
        time_dim = precip_ds.dims.get('time', len(grib_files))
        print(f"  Time steps: {time_dim}")
        print(f"  Dataset:\n{precip_ds}")
    except Exception as e:
        print(f"Extraction requires cfgrib/xarray: {e}")
        print("\nInstall with: pip install cfgrib eccodes")
else:
    print("Skipping extraction (no GRIB files downloaded)")
    print("\nExpected output:")
    print("  Variables: ['APCP']")
    print("  Dimensions: {'time': 18, 'latitude': 36, 'longitude': 72}")
    print("  APCP units: kg m-2 (equivalent to mm)")

2026-06-24 22:06:22 - ras_commander.precip.PrecipHrrr - INFO - Reading 18 HRRR GRIB2 files


2026-06-24 22:06:32 - ras_commander.precip.PrecipHrrr - INFO - Clipped to bounds: (-77.9, 40.8, -77.3, 41.1)


2026-06-24 22:06:32 - ras_commander.precip.PrecipHrrr - INFO - Extracted precipitation from 18 files


Extracted precipitation dataset:
  Variables: ['tp']
  Dimensions: {'step': 72, 'y': 15, 'x': 18}
  Time steps: 18
  Dataset:
<xarray.Dataset> Size: 83kB
Dimensions:     (step: 72, y: 15, x: 18)
Coordinates:
  * step        (step) timedelta64[ns] 576B 00:15:00 00:30:00 ... 18:00:00
    valid_time  (step) datetime64[ns] 576B 2026-06-25T00:15:00 ... 2026-06-25...
    latitude    (y, x) float64 2kB 40.81 40.81 40.8 40.8 ... 41.1 41.09 41.08
    longitude   (y, x) float64 2kB 282.1 282.1 282.1 282.2 ... 282.7 282.7 282.8
    time        datetime64[ns] 8B 2026-06-25
    surface     float64 8B 0.0
Dimensions without coordinates: y, x
Data variables:
    tp          (step, y, x) float32 78kB nan nan 0.0 nan ... nan 0.0 nan nan
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCE

C:\Users\bill\AppData\Local\Temp\ipykernel_42824\3606873314.py:12: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f"  Dimensions: {dict(precip_ds.dims)}")
C:\Users\bill\AppData\Local\Temp\ipykernel_42824\3606873314.py:13: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  time_dim = precip_ds.dims.get('time', len(grib_files))


### Basin-Average Precipitation

`get_basin_average()` spatially averages the gridded HRRR precipitation over a watershed geometry and returns a `pandas.DataFrame` with hourly precipitation and cumulative totals. The geometry can be any Shapely geometry (polygon, multipolygon, etc.).

The returned DataFrame has columns:
- `forecast_hour` - Hours from cycle start (1 through N)
- `precip_mm` - Incremental precipitation in millimeters
- `precip_inches` - Incremental precipitation in inches
- `cumulative_mm` - Cumulative precipitation in millimeters
- `cumulative_inches` - Cumulative precipitation in inches

In [7]:
if grib_files:
    try:
        from shapely.geometry import box
        basin_geom = box(-77.9, 40.8, -77.3, 41.1)

        avg_precip = PrecipHrrr.get_basin_average(
            grib_files=grib_files,
            geometry=basin_geom
        )
        print("Basin-average precipitation time series:")
        print(avg_precip.to_string(index=False))
        print(f"\nTotal forecast precipitation: {avg_precip['cumulative_inches'].iloc[-1]:.3f} inches")
        print(f"Peak hourly precipitation:    {avg_precip['precip_inches'].max():.3f} inches/hr")
    except Exception as e:
        print(f"Basin average calculation failed: {e}")
else:
    print("Skipping basin average (no GRIB files)")
    print("\nExpected output (DataFrame columns):")
    print("  forecast_hour | precip_mm | precip_inches | cumulative_mm | cumulative_inches")
    print("  1             | 0.5       | 0.02          | 0.5           | 0.02")
    print("  2             | 1.2       | 0.05          | 1.7           | 0.07")
    print("  ...")

2026-06-24 22:06:32 - ras_commander.precip.PrecipHrrr - INFO - Reading 18 HRRR GRIB2 files


2026-06-24 22:06:34 - ras_commander.precip.PrecipHrrr - INFO - Extracted precipitation from 18 files


2026-06-24 22:06:34 - ras_commander.precip.PrecipHrrr - INFO - Calculating basin-average precipitation


2026-06-24 22:06:34 - ras_commander.precip.PrecipHrrr - INFO - Basin average: 0.00 inches total over 72 hours


Basin-average precipitation time series:
 forecast_hour  precip_mm  precip_inches  cumulative_mm  cumulative_inches
             1   0.000000   0.000000e+00       0.000000           0.000000
             2   0.000000   0.000000e+00       0.000000           0.000000
             3   0.000000   0.000000e+00       0.000000           0.000000
             4   0.000000   0.000000e+00       0.000000           0.000000
             5   0.000000   0.000000e+00       0.000000           0.000000
             6   0.000000   0.000000e+00       0.000000           0.000000
             7   0.000000   0.000000e+00       0.000000           0.000000
             8   0.000000   0.000000e+00       0.000000           0.000000
             9   0.000000   0.000000e+00       0.000000           0.000000
            10   0.000000   0.000000e+00       0.000000           0.000000
            11   0.000000   0.000000e+00       0.000000           0.000000
            12   0.000000   0.000000e+00       0.000000    

## Example 2: Specific Date/Cycle Download

When you need a specific historical cycle (within NOMADS retention window, typically ~48 hours), use `download_forecast()` with explicit `date` and `cycle` parameters. This is useful for:
- Comparing model forecasts to observations after an event
- Reproducible analysis with a known dataset
- Operational post-processing pipelines

In [8]:
# Download a specific date and cycle
output_dir_specific = Path("example_data/hrrr_specific")
output_dir_specific.mkdir(parents=True, exist_ok=True)

# Use yesterday's 00z cycle with 18-hour forecast
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

try:
    grib_files_specific = PrecipHrrr.download_forecast(
        output_dir=output_dir_specific,
        date=yesterday,
        cycle=0,   # 00z cycle
        hours=18,
        overwrite=False  # Skip if already downloaded
    )
    print(f"Downloaded {len(grib_files_specific)} files for {yesterday} 00z")
    if grib_files_specific:
        total_mb = sum(f.stat().st_size for f in grib_files_specific) / (1024 * 1024)
        print(f"Total download size: {total_mb:.1f} MB")
except Exception as e:
    print(f"Download requires internet: {e}")
    print(f"\nWould download 18 GRIB2 files for {yesterday} 00z cycle")
    print("Typical file size: ~100-150 MB per GRIB2 file (full CONUS)")
    print("Total download: ~2-3 GB for full 18-hour cycle")

2026-06-24 22:06:34 - ras_commander.precip.PrecipHrrr - INFO - Downloading: hrrr.t00z.wrfsubhf01.grib2 (hour 1/18)


Download requires internet: Cannot reach NOMADS server. Check internet connection. Error: ('Connection aborted.', BadStatusLine('HTTP/1.1   0 Init\r\n'))

Would download 18 GRIB2 files for 2026-06-23 00z cycle
Typical file size: ~100-150 MB per GRIB2 file (full CONUS)
Total download: ~2-3 GB for full 18-hour cycle


### HRRR Cycle Timing Reference

Understanding HRRR cycle timing is critical for operational use. The following cell provides a reference table for all 24 daily cycles.

In [9]:
# HRRR cycles and forecast horizons (all 24 hourly cycles)
print("HRRR Forecast Cycles:")
print("=" * 65)
print(f"{'Cycle':<10} {'Type':<12} {'Forecast Hours':<20} {'Approx Availability (UTC)'}")
print("-" * 65)

EXTENDED_CYCLES = [0, 6, 12, 18]

for cycle in range(0, 24):
    if cycle in EXTENDED_CYCLES:
        cycle_type = "Extended"
        hours = "48 hours"
    else:
        cycle_type = "Standard"
        hours = "18 hours"
    avail_hour = (cycle + 2) % 24
    availability = f"{avail_hour:02d}:00 - {avail_hour:02d}:30 UTC"
    print(f"{cycle:02d}z       {cycle_type:<12} {hours:<20} {availability}")

print()
print("Notes:")
print("  - HRRR runs every hour (24 cycles per day, 00z through 23z)")
print("  - Data typically available ~2 hours after cycle start time")
print("  - Example: 12z cycle data available ~14:00 UTC")
print("  - Extended cycles (00/06/12/18z) produce 48-hour forecasts")
print("  - All other cycles produce 18-hour forecasts")
print("  - NOMADS retains ~48 hours of recent cycles")
print("  - For older data: https://storage.googleapis.com/high-resolution-rapid-refresh/")
print()
print("File size reference (full CONUS wrfsubhf):")
print("  ~100-150 MB per GRIB2 file (single forecast hour)")
print("  ~2-3 GB for complete 18-hour standard cycle")
print("  ~5-7 GB for complete 48-hour extended cycle")
print("  Spatial subsetting via 'bounds' parameter reduces size significantly")

HRRR Forecast Cycles:
Cycle      Type         Forecast Hours       Approx Availability (UTC)
-----------------------------------------------------------------
00z       Extended     48 hours             02:00 - 02:30 UTC
01z       Standard     18 hours             03:00 - 03:30 UTC
02z       Standard     18 hours             04:00 - 04:30 UTC
03z       Standard     18 hours             05:00 - 05:30 UTC
04z       Standard     18 hours             06:00 - 06:30 UTC
05z       Standard     18 hours             07:00 - 07:30 UTC
06z       Extended     48 hours             08:00 - 08:30 UTC
07z       Standard     18 hours             09:00 - 09:30 UTC
08z       Standard     18 hours             10:00 - 10:30 UTC
09z       Standard     18 hours             11:00 - 11:30 UTC
10z       Standard     18 hours             12:00 - 12:30 UTC
11z       Standard     18 hours             13:00 - 13:30 UTC
12z       Extended     48 hours             14:00 - 14:30 UTC
13z       Standard     18 hours    

## Example 3: Writing HRRR Forecast to DSS with qpkit (Optional)

This optional section demonstrates how to write HRRR precipitation forecasts directly to a
HEC-DSS file using **qpkit** (by Gyan Basyal / WEST Consultants). qpkit fills a genuine gap
in the ras-commander HRRR workflow: ras-commander can download HRRR GRIB2 files and compute
basin-average time series (Examples 1 and 2 above), but it does not currently provide a
native path for writing gridded HRRR precipitation to DSS. qpkit does this in a single
`download_to_dss()` call via its `pydsstools` backend.

**Attribution**: qpkit is an open-source library (Apache-2.0) authored by Gyan Basyal at
WEST Consultants. Repository: https://github.com/gyanz/qpkit — pydsstools:
https://github.com/gyanz/pydsstools

**When to use qpkit vs the native ras-commander path**:

| Need | Use |
|------|-----|
| Download HRRR + extract basin-average time series | `PrecipHrrr` (this notebook, Examples 1-2) |
| Write gridded HRRR precipitation to HEC-DSS | `qpkit.QPKit` + `HRRRGridOptions` (this section) |
| Verify DSS pathnames after writing | `RasDss.get_catalog()` (shown below) |

All cells in this section are gated behind a `HAVE_QPKIT` flag so the notebook runs
end-to-end on environments that do not have qpkit installed.

In [10]:
# Optional dependency: qpkit (by Gyan Basyal / WEST Consultants)
# Install the pinned commit with:
#   pip install "git+https://github.com/gyanz/qpkit.git@efbe1eb13fbad3ad2ed2ac3f73bf9c3af6839ecc"
#
# qpkit depends transitively on pydsstools>=3.1.0.

try:
    import qpkit  # noqa: F401
    HAVE_QPKIT = True
    print(f"qpkit {qpkit.__version__} found — DSS write cells will execute.")
except ImportError:
    HAVE_QPKIT = False
    print(
        "qpkit not installed — Example 3 cells will be skipped.\n"
        "To enable, run:\n"
        '  pip install "git+https://github.com/gyanz/qpkit.git'
        '@efbe1eb13fbad3ad2ed2ac3f73bf9c3af6839ecc"\n'
        "(requires pydsstools>=3.1.0)"
    )


2026-06-24 22:06:35 - pydsstools.heclib.logging - INFO - DssLogger(GLOBAL, level=TERSE): level set — method=GLOBAL, level=TERSE.


qpkit 0.0.1.dev33+gefbe1eb13 found — DSS write cells will execute.


In [11]:
if HAVE_QPKIT:
    from datetime import date, datetime, timedelta, timezone
    from pathlib import Path
    from pydsstools.core.crs import shg
    from qpkit import BoundingBox, HRRRRequest, QPKit
    from qpkit.models import HRRRGridOptions

    # Reuse the same bounding box as Examples 1 and 2: Bald Eagle Creek, PA.
    HRRR_BBOX = BoundingBox(
        left_lon=-77.9,
        right_lon=-77.3,
        top_lat=41.1,
        bottom_lat=40.8,
    )

    # HRRR is retained on NOMADS for only ~2 days, so a frozen date cannot
    # download live. Use a recent cycle (yesterday 00Z UTC) so this cell runs.
    CYCLE_DATE = (datetime.now(timezone.utc) - timedelta(days=1)).date()
    CYCLE_HOUR = 0
    # forecast_hours=(6,) requests up to hour 6.
    # download_to_dss(interval=1) differences consecutive APCP grids
    # to produce per-hour accumulations automatically.
    FORECAST_HOURS = (6,)

    qpkit_output_dir = Path("example_data/hrrr_qpkit")
    qpkit_output_dir.mkdir(parents=True, exist_ok=True)
    dss_file = qpkit_output_dir / "hrrr_precip.dss"

    request = HRRRRequest(
        precip_type="APCP",
        cycle_date=CYCLE_DATE,
        cycle_hour=CYCLE_HOUR,
        forecast_hours=FORECAST_HOURS,
        bbox=HRRR_BBOX,
    )

    options = HRRRGridOptions.for_apcp(
        part_a="HRRR",
        part_b="BALD_EAGLE_CRK",  # Name the clipped region, not CONUS
        part_c="PRECIP",
        part_f="HRRR-APCP-1H",
        crs=shg(),        # SHG CRS — standard for HEC-HMS/HEC-RAS grids
        cell_size=3000,   # 3 km native HRRR resolution in meters
    )

    try:
        with QPKit() as kit:
            result = kit.download_to_dss(
                request,
                qpkit_output_dir,   # Folder where GRIB2 files are staged
                dss_file,           # Output HEC-DSS file
                grid_options=options,
                dss_version=6,
                interval=1,         # Write 1-hour incremental accumulation grids
                force=True,         # Re-download even if GRIB is already on disk
            )

        print(f"GRIB2 files downloaded : {len(result.download.succeeded)}")
        print(f"DSS grids written      : {len(result.dss.written)}")
        print(f"DSS grids skipped      : {len(result.dss.skipped)}")
        print(f"DSS grids failed       : {len(result.dss.failed)}")
        print(f"Output DSS file        : {dss_file}")
        print()
        print("DSS pathnames written:")
        for pathname in result.dss.all_pathnames:
            print(f"  {pathname}")
    except Exception as e:
        print(f"qpkit download_to_dss failed (requires network + NOMADS access): {e}")
        print()
        print("Expected output:")
        print("  GRIB2 files downloaded : 7")
        print("  DSS grids written      : 6")
        print("  DSS pathnames written:")
        print("  /HRRR/BALD_EAGLE_CRK/PRECIP/08MAY2026:0500/08MAY2026:0600/HRRR-APCP-1H/")
        print("  /HRRR/BALD_EAGLE_CRK/PRECIP/08MAY2026:0600/08MAY2026:0700/HRRR-APCP-1H/")
        print("  ...")
else:
    print("Skipping qpkit DSS write (HAVE_QPKIT=False).")
    dss_file = None


2026-06-24 22:06:35 - httpx - INFO - HTTP Request: GET https://nomads.ncep.noaa.gov/cgi-bin/filter_hrrr_2d.pl?file=hrrr.t00z.wrfsfcf00.grib2&lev_surface=on&var_APCP=on&subregion=&leftlon=-77.9&rightlon=-77.3&toplat=41.1&bottomlat=40.8&dir=%2Fhrrr.20260624%2Fconus "HTTP/1.1 200 OK"


2026-06-24 22:06:35 - qpkit - INFO - Download saved to: example_data\hrrr_qpkit\hrrr.20260624.t00z.wrfsfc.f00.APCP.grib2


2026-06-24 22:06:35 - httpx - INFO - HTTP Request: GET https://nomads.ncep.noaa.gov/cgi-bin/filter_hrrr_2d.pl?file=hrrr.t00z.wrfsfcf01.grib2&lev_surface=on&var_APCP=on&subregion=&leftlon=-77.9&rightlon=-77.3&toplat=41.1&bottomlat=40.8&dir=%2Fhrrr.20260624%2Fconus "HTTP/1.1 200 OK"


2026-06-24 22:06:35 - qpkit - INFO - Download saved to: example_data\hrrr_qpkit\hrrr.20260624.t00z.wrfsfc.f01.APCP.grib2


2026-06-24 22:06:36 - httpx - INFO - HTTP Request: GET https://nomads.ncep.noaa.gov/cgi-bin/filter_hrrr_2d.pl?file=hrrr.t00z.wrfsfcf03.grib2&lev_surface=on&var_APCP=on&subregion=&leftlon=-77.9&rightlon=-77.3&toplat=41.1&bottomlat=40.8&dir=%2Fhrrr.20260624%2Fconus "HTTP/1.1 200 OK"


...

2026-06-24 22:06:37 - pydsstools._lib.x64.core_heclib - INFO - Size of compressed data: 33 bytes


2026-06-24 22:06:37 - pydsstools._lib.x64.core_heclib - INFO - Length of compressed data: 9


2026-06-24 22:06:37 - pydsstools._lib.x64.core_heclib - INFO - Grid data written to file with status code = 0


2026-06-24 22:06:37 - qpkit.services.dss.hrrr - INFO - [6/6] hrrr.20260624.t00z.wrfsfc.f05.APCP.grib2 → hrrr.20260624.t00z.wrfsfc.f06.APCP.grib2  done


GRIB2 files downloaded : 7
DSS grids written      : 6
DSS grids skipped      : 0
DSS grids failed       : 0
Output DSS file        : example_data\hrrr_qpkit\hrrr_precip.dss

DSS pathnames written:
  /HRRR/BALD_EAGLE_CRK/PRECIP/24Jun2026:0000/24Jun2026:0100/HRRR-APCP-1H/
  /HRRR/BALD_EAGLE_CRK/PRECIP/24Jun2026:0100/24Jun2026:0200/HRRR-APCP-1H/
  /HRRR/BALD_EAGLE_CRK/PRECIP/24Jun2026:0200/24Jun2026:0300/HRRR-APCP-1H/
  /HRRR/BALD_EAGLE_CRK/PRECIP/24Jun2026:0300/24Jun2026:0400/HRRR-APCP-1H/
  /HRRR/BALD_EAGLE_CRK/PRECIP/24Jun2026:0400/24Jun2026:0500/HRRR-APCP-1H/
  /HRRR/BALD_EAGLE_CRK/PRECIP/24Jun2026:0500/24Jun2026:0600/HRRR-APCP-1H/


In [12]:
# Optional: read back the written DSS pathnames via ras-commander's RasDss.
# This verifies interoperability between qpkit's pydsstools DSS write and
# ras-commander's Java-based DSS bridge.

if HAVE_QPKIT and dss_file is not None and dss_file.exists():
    try:
        from ras_commander.dss import RasDss

        catalog = RasDss.get_catalog(dss_file)
        print(f"RasDss.get_catalog() found {len(catalog)} pathnames in {dss_file.name}:")
        print(catalog.to_string(index=False))
    except Exception as e:
        # RasDss requires the HEC-Monolith Java bridge (pyjnius + JVM).
        # Skip gracefully if the JVM is not available in this environment.
        print(f"RasDss.get_catalog() skipped (requires JVM / pyjnius): {e}")
        print(
            "\nExpected output (one row per 1-hour APCP interval written):\n"
            "  pathname\n"
            "  /HRRR/BALD_EAGLE_CRK/PRECIP/08MAY2026:0500/08MAY2026:0600/HRRR-APCP-1H/\n"
            "  /HRRR/BALD_EAGLE_CRK/PRECIP/08MAY2026:0600/08MAY2026:0700/HRRR-APCP-1H/\n"
            "  ..."
        )
else:
    print("Skipping RasDss catalog check (qpkit not installed or DSS file not created).")


Configuring Java VM for DSS operations...
[OK] Java VM configured


RasDss.get_catalog() found 6 pathnames in hrrr_precip.dss:
                                                               pathname
/HRRR/BALD_EAGLE_CRK/PRECIP/24JUN2026:0000/24JUN2026:0100/HRRR-APCP-1H/
/HRRR/BALD_EAGLE_CRK/PRECIP/24JUN2026:0100/24JUN2026:0200/HRRR-APCP-1H/
/HRRR/BALD_EAGLE_CRK/PRECIP/24JUN2026:0200/24JUN2026:0300/HRRR-APCP-1H/
/HRRR/BALD_EAGLE_CRK/PRECIP/24JUN2026:0300/24JUN2026:0400/HRRR-APCP-1H/
/HRRR/BALD_EAGLE_CRK/PRECIP/24JUN2026:0400/24JUN2026:0500/HRRR-APCP-1H/
/HRRR/BALD_EAGLE_CRK/PRECIP/24JUN2026:0500/24JUN2026:0600/HRRR-APCP-1H/


## Key Takeaways

**HRRR Data Characteristics**:
- 3km resolution, hourly cycles, publicly available from NOAA NOMADS
- Standard cycles produce 18-hour forecasts; extended cycles (00/06/12/18z) produce 48-hour forecasts
- NOMADS retains approximately 48 hours of recent output; use the HRRR archive on AWS for older data

**Workflow Summary**:
1. Use `check_availability()` to verify data is on NOMADS before downloading
2. Use `get_latest_forecast()` for operational workflows that always need current data
3. Use `download_forecast(date=..., cycle=...)` when you need a specific historical cycle
4. Use `extract_precipitation(bounds=...)` to clip to your region of interest (reduces memory)
5. Use `get_basin_average()` to compute watershed-averaged precipitation for HEC-RAS boundary conditions

**Common Pitfalls**:
- HRRR precipitation (`APCP`) is accumulated from the start of the cycle — convert to incremental depth before summing
- The `overwrite=False` default prevents redundant downloads in operational pipelines
- For production use, consider caching GRIB2 files locally; full CONUS files are ~100-150 MB each (~2-3 GB per 18-hour cycle)
- cfgrib and eccodes are required for `extract_precipitation()`: `pip install cfgrib eccodes`

**Integration with HEC-RAS**:
- Use `get_basin_average()` output as upstream boundary condition hyetographs
- Combine with `RasUnsteady.set_precipitation_hyetograph()` to write directly to unsteady flow files
- For spatially distributed rainfall, export `extract_precipitation()` output to DSS using `RasDss`

## Cleanup

In [13]:
import shutil

for d in ["example_data/hrrr_single", "example_data/hrrr_specific", "example_data/hrrr_qpkit"]:
    p = Path(d)
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)
        print(f"Cleaned up: {d}")
    else:
        print(f"Not found (nothing to clean): {d}")

print("Done!")


Cleaned up: example_data/hrrr_single
Cleaned up: example_data/hrrr_specific
Cleaned up: example_data/hrrr_qpkit
Done!


## Acknowledgements

This workflow is powered by **[qpkit](https://github.com/gyanz/qpkit)**, an open-source library (Apache-2.0) by **[Gyan Basyal](https://github.com/gyanz)** of WEST Consultants, Inc. qpkit builds directly on **[pydsstools](https://github.com/gyanz/pydsstools)** (also authored by Gyan Basyal), extending it with NOAA precipitation acquisition (MRMS QPE, WPC QPF, HRRR) and direct GRIB2-to-HEC-DSS grid writing.

Please see and support his work:

- **qpkit** (used in this notebook): https://github.com/gyanz/qpkit
- **Gyan Basyal** on GitHub: https://github.com/gyanz
- **pydsstools** (the HEC-DSS engine qpkit builds on): https://github.com/gyanz/pydsstools
